# CarDex — make/model training (Kaggle)

1. Attach the **Car Model Variants and Images** dataset (Add Input).
2. Settings → Accelerator → **GPU** (T4 x2 or P100).
3. Run the cells top to bottom. Outputs land in `/kaggle/working` — **Save Version** to keep them.

Getting the scaffold: easiest is to `git clone` your repo (cell below). No repo yet? Upload
`recognition/training/` as a Kaggle dataset and add it as input, or paste the three `.py` files
into cells with `%%writefile`.

In [ ]:
# GPU + dataset sanity check
!nvidia-smi -L
!ls -la /kaggle/input

In [ ]:
# Get the training scaffold (edit to your repo), then install timm.
# torch / torchvision / pandas / scikit-learn are preinstalled on Kaggle.
!git clone https://github.com/YOUR_USERNAME/cardex.git
%cd cardex/recognition/training
!pip -q install timm

In [ ]:
# Point this at the mounted dataset (confirm the exact folder name from the ls above).
DATA = "/kaggle/input/car-model-variants-and-images-dataset"
!ls $DATA   # expect: train/  test/  dataset.csv

In [ ]:
# Build the manifest + label space. CHECK THE PRINTED MATCH RATE.
!python build_manifest.py \
  --data-dir $DATA \
  --metadata $DATA/dataset.csv \
  --out-dir /kaggle/working/out
# If the match rate is low, inspect the misses:
# !head -40 /kaggle/working/out/unmatched.txt

In [ ]:
# Smoke test — 1 epoch to prove the whole pipeline runs before a long job.
!python train.py \
  --manifest /kaggle/working/out/manifest.csv \
  --classes  /kaggle/working/out/classes.json \
  --out-dir  /kaggle/working/model \
  --epochs 1 --batch-size 32

In [ ]:
# Full run. Drop --batch-size / --img-size if you hit CUDA OOM.
!python train.py \
  --manifest /kaggle/working/out/manifest.csv \
  --classes  /kaggle/working/out/classes.json \
  --out-dir  /kaggle/working/model \
  --backbone tf_efficientnet_b0 --epochs 12 --batch-size 64

In [ ]:
# Artifacts to download / Save Version: model.pt + embeddings.npz (+ catalogue.csv)
!ls -la /kaggle/working/model
!ls -la /kaggle/working/out/catalogue.csv

## Next
- Download `model.pt` + `embeddings.npz` and serve them via `recognition/classifier.py`
  (`MakeModelClassifier.from_files(...)`).
- The real accuracy check is a handful of **your own phone photos** — that's the domain-gap test.
- Ingest `catalogue.csv` into the `cars` table so the model's labels map to real `car_id`s.